# ArchiGen 3D — Run 2 on Kaggle (recipe v2, ~10 h)

**Goal:** close the gap with the historical models using their training recipe, measured with the same protocol.

| GPU | Job | Recipe v2 changes | Cap |
|---|---|---|---|
| 0 | **Pix2Pix**, 100 epochs, then **U-Net L1 baseline**, 100 epochs | real LPIPS loss (SqueezeNet), **EMA** generator, constant→linear LR decay, crop jitter, checkpoint chosen on **val LPIPS** (no L1 early stopping) | 6 h + 3 h |
| 1 | **UNet++ floor plans**, **512 px**, 25 epochs, mixed precision | **ImageNet ResNet-34 encoder**, cosine LR, room class weights (rare rooms), batch 4 × 2 | 10 h |

Evaluation (test split, once): facades → L1, PSNR, SSIM, LPIPS (AlexNet **and** SqueezeNet), **KID ×1000**;
floor plans → single pass **and** 4-fold TTA.

**Before running:** *Session options* → **Accelerator: GPU T4 ×2**, **Internet: On** (pretrained weights are downloaded).
**+ Add Input**: your code dataset (re-upload `archigen_facades.zip` built from the **current** code — it contains recipe v2)
and the public **CubiCasa5k** dataset (rasterized automatically). Then **Save Version → Save & Run All**.
A session lasts at most **12 h**; this run uses ~10 h of your 30 h weekly GPU quota.

In [ ]:
# 1) Setup: find the inputs, build a writable workspace, install metrics, check GPUs
import glob, os, shutil, subprocess, sys, time, json
from pathlib import Path

def find(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return Path(hits[0]) if hits else None

code_root = find('/kaggle/input/**/archigen/params.yaml')
FACADES = find('/kaggle/input/**/archigen/data/raw/facades')
CUBICASA = find('/kaggle/input/**/archigen/data/processed/cubicasa_cache')
assert code_root, "Code not found: '+ Add Input' -> the dataset made from archigen_facades.zip"
SRC = code_root.parent
assert (SRC / 'src/models/unet_plus_plus.py').read_text().count('load_imagenet_resnet34'), \
    'This code bundle predates recipe v2: rebuild it with `python kaggle/make_bundle.py --part facades` and re-upload.'

W = Path('/kaggle/working/archigen')
if W.exists():
    shutil.rmtree(W)
(W / 'data/raw').mkdir(parents=True)
(W / 'data/processed').mkdir(parents=True)
shutil.copytree(SRC / 'src', W / 'src')
shutil.copy(SRC / 'params.yaml', W / 'params.yaml')
if FACADES:
    os.symlink(FACADES, W / 'data/raw/facades')
if CUBICASA:
    os.symlink(CUBICASA, W / 'data/processed/cubicasa_cache')
else:
    raw = [Path(t).parent for t in glob.glob('/kaggle/input/**/train.txt', recursive=True)
           if (Path(t).parent / 'val.txt').exists() and (Path(t).parent / 'test.txt').exists()]
    if raw:
        print('Rasterizing CubiCasa5K from', raw[0], '(official splits)...')
        r = subprocess.run([sys.executable, '-m', 'src.data.rasterize_cubicasa', '--source', str(raw[0]),
                            '--out', str(W / 'data/processed/cubicasa_cache'), '--workers', str(os.cpu_count() or 4)],
                           cwd=W, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
HAS_FACADES = FACADES is not None
HAS_FP = any((W / 'data/processed/cubicasa_cache/train').glob('*.npz')) if (W / 'data/processed/cubicasa_cache/train').exists() else False
os.chdir(W)

# LPIPS (loss + metric) and KID (torchmetrics + torch-fidelity); torch/torchvision are preinstalled on Kaggle.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lpips', 'torchmetrics[image]'])

import torch
N_GPU = torch.cuda.device_count()
print('torch', torch.__version__, '| GPUs:', [torch.cuda.get_device_name(i) for i in range(N_GPU)])
print('facades:', HAS_FACADES, '| cubicasa:', HAS_FP)
assert N_GPU > 0, 'No GPU: Session options > Accelerator > GPU T4 x2'

In [ ]:
# 2) Train (blocks until done; progress every 2 minutes)
PY = sys.executable
P2P = f'{PY} -m src.training.train_facade --mode pix2pix --recipe v2 --epochs 100 --workers 2 --time-budget-min 360'
L1  = f'{PY} -m src.training.train_facade --mode l1_baseline --recipe v2 --epochs 100 --workers 2 --time-budget-min 180'
FP  = f'{PY} -m src.training.train_floorplan --recipe v2 --image-size 512 --amp --epochs 25 --workers 3 --time-budget-min 600'

lanes = {}
if HAS_FACADES:
    lanes['facades'] = (0, f'{P2P} ; {L1}')
if HAS_FP:
    lanes['floorplan'] = (1 if N_GPU >= 2 and HAS_FACADES else 0, FP)
if N_GPU < 2 and len(lanes) == 2:   # single GPU: run in sequence, floor plans first get a smaller cap
    lanes = {'all': (0, f"{FP.replace('--time-budget-min 600', '--time-budget-min 300')} ; {P2P.replace('360', '200')} ; {L1.replace('180', '90')}")}

procs = {}
for name, (gpu, cmd) in lanes.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED='1',
               MLFLOW_TRACKING_URI=f'sqlite:///{W}/mlflow_{name}.db')
    procs[name] = subprocess.Popen(cmd, shell=True, cwd=W, env=env,
                                   stdout=open(W / f'log_{name}.txt', 'w'), stderr=subprocess.STDOUT)
    print(f'started {name} on GPU {gpu}: {cmd}')

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    print(f'--- {(time.time() - t0) / 60:.0f} min ---')
    for name in procs:
        lines = (W / f'log_{name}.txt').read_text(errors='ignore').splitlines()
        keep = [l for l in lines if any(k in l for k in ('Epoch', 'budget', 'Error', 'mode=', 'Floorplan]'))]
        print(f'  {name}: {keep[-1] if keep else (lines[-1] if lines else "starting...")}')
print('exit codes:', {n: p.returncode for n, p in procs.items()}, f'| {(time.time() - t0) / 60:.1f} min')
for name, p in procs.items():
    if p.returncode != 0:
        print(f'\n===== last lines of log_{name}.txt =====')
        print('\n'.join((W / f'log_{name}.txt').read_text(errors='ignore').splitlines()[-30:]))

In [ ]:
# 3) Final evaluation on the TEST split (once)
def run(*args):
    subprocess.run([sys.executable, '-m', 'src.evaluation.evaluate', *args], cwd=W)

ck = W / 'models/checkpoints'
for name in ('facade_pix2pix_best.pt', 'facade_unet_l1_best.pt'):
    print(f'\n===== {name} ====='); run('facade', '--checkpoint', str(ck / name)) if (ck / name).exists() else print('MISSING')
fp = ck / 'floorplan_unetpp_best.pt'
if fp.exists():
    print('\n===== floorplan, single pass ====='); run('floorplan', '--checkpoint', str(fp), '--batch-size', '4')
    print('\n===== floorplan, 4-fold TTA ====='); run('floorplan', '--checkpoint', str(fp), '--batch-size', '4', '--tta')

In [ ]:
# 4) Run 2 vs historical reference (numbers from the Nov 2026 model-evaluation report, CMP / CubiCasa test splits)
M = W / 'models/metrics'
load = lambda n: json.loads((M / n).read_text()) if (M / n).exists() else None
mean = lambda d, k: (d.get(k) or {}).get('mean') if d else None
fmt = lambda v, p=4: '—' if v is None else f'{v:.{p}f}'

print('FACADES (test, 106 images)            L1       PSNR    SSIM    LPIPS-sq  KIDx1000')
print('  reference: EMA (historical)         0.3403   12.97   0.2893  0.3096    63.55')
for name in ('facade_pix2pix_best_test.json', 'facade_unet_l1_best_test.json'):
    r = load(name)
    print(f'  run 2: {name[:-15]:<28}', fmt(mean(r, 'l1')), ' ', fmt(mean(r, 'psnr_db'), 2), ' ', fmt(mean(r, 'ssim')),
          fmt(mean(r, 'lpips_squeeze')), '  ', fmt(mean(r, 'kid_x1000'), 2))

def fp_row(r):
    if not r: return None
    s, rooms = r['structure']['per_class_iou'], r['rooms']['per_class_iou']
    no_bg = [v for k, v in rooms.items() if k != 'background' and v is not None]
    return [100 * r['merged_wall_iou'], 100 * (s['door'] or 0), 100 * (s['window'] or 0),
            100 * sum(no_bg) / max(1, len(no_bg)), 100 * r['rooms']['pixel_acc']]

print('\nFLOOR PLANS (test, 400 plans)        wall    door    window  room mIoU*  room acc')
print('  reference single pass               74.17   61.78   73.52   56.40       87.50')
print('  reference 4-fold TTA                74.93   63.21   74.65   58.24       88.09')
for label, name in (('run 2 single pass', 'floorplan_unetpp_best_test.json'), ('run 2 4-fold TTA', 'floorplan_unetpp_best_test_tta.json')):
    row = fp_row(load(name))
    print(f'  {label:<34}', '   '.join('—' if row is None else f'{v:5.2f}' for v in (row or [None] * 5)))
print('\n* room mIoU: reference = 10 CubiCasa classes; run 2 = 11 ArchiGen room classes without background.'
      ' Class sets differ, so treat that column as indicative. Wall/door/window IoU use the same definitions.')

In [ ]:
# 5) Package results -> /kaggle/working/archigen_results_run2.zip (download from the Output tab)
R = Path('/kaggle/working/archigen_results_run2')
if R.exists():
    shutil.rmtree(R)
(R / 'checkpoints').mkdir(parents=True)
for f in (W / 'models/checkpoints').glob('*_best.pt'):
    shutil.copy(f, R / 'checkpoints')
shutil.copytree(W / 'models/metrics', R / 'metrics')
if (W / 'models/samples').exists():
    shutil.copytree(W / 'models/samples', R / 'samples')
for f in list(W.glob('log_*.txt')) + list(W.glob('mlflow_*.db')):
    shutil.copy(f, R)
shutil.make_archive('/kaggle/working/archigen_results_run2', 'zip', R)
shutil.rmtree(R)
shutil.rmtree(W)
print('Ready: /kaggle/working/archigen_results_run2.zip',
      f"{Path('/kaggle/working/archigen_results_run2.zip').stat().st_size / 1e6:.0f} MB")